In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from web3 import Web3
import requests
import sys
sys.path.append('/Users/noahturner/code/Blockchain-Project/oracle')
from odds_client import get_client

load_dotenv(find_dotenv())

contract_address = os.getenv("CONTRACT_ADDRESS")
rpc_url = os.getenv("RPC_URL")
etherscan_api_key = os.getenv("ETHERSCAN_API_KEY")
chain_id = os.getenv("NEXT_PUBLIC_CHAIN_ID")
odds_api_key = os.getenv("SPORTS_API_KEY")
bettor = "0x244e18F228b2804f3a4445d2446c1847b55B294A"
# getenv returns str | None; function also accepts int or str
first_block = int(os.getenv("CONTRACT_DEPLOY_BLOCK") or "0")






In [27]:

# def get_raw_logs():
#     contract_address = os.getenv("CONTRACT_ADDRESS")
#     rpc_url = os.getenv("RPC_URL")
#     etherscan_api_key = os.getenv("ETHERSCAN_API_KEY")
chain_id = os.getenv("NEXT_PUBLIC_CHAIN_ID")
resp = requests.get(
    f"https://api.etherscan.io/v2/api?chainid={chain_id}&module=proxy&action=eth_blockNumber&apikey={etherscan_api_key}"
).json()
latest_block_hex = resp["result"]      # e.g. "0xa2d1f3"
latest_block = int(latest_block_hex, 16)
response = requests.get(f"https://api.etherscan.io/v2/api?chainid={chain_id}&module=logs&action=getLogs&address={contract_address}&fromBlock={first_block}&toBlock={latest_block}&page=1&offset=100&apikey={etherscan_api_key}")

logs = response.json()['result']


logs

[{'address': '0xd0f5021a1e3fa57206cd09a341d0b2859bd55da7',
  'topics': ['0x5af8184bef8e4b45eb9f6ed7734d04da38ced226495548f46e0c8ff8d7d9a524',
   '0x000000000000000000000000af45f76c18d8ae423c5912911dcb8929ec3bbd88'],
  'data': '0x00000000000000000000000000000000000000000000000000006fde2b4eb000',
  'blockNumber': '0xa2b301',
  'blockHash': '0xef228d4c82cd39574a1de7ceebb5766c807db2615f61e72a52b773d226abbc41',
  'timeStamp': '0x69df194c',
  'gasPrice': '0xa1bbfb59',
  'gasUsed': '0x5861',
  'logIndex': '0x25',
  'transactionHash': '0xf6a0649a6bf951034bd78290ef5a56c556cfa15561f49756009c16baa6ca5395',
  'transactionIndex': '0x11'},
 {'address': '0xd0f5021a1e3fa57206cd09a341d0b2859bd55da7',
  'topics': ['0x5af8184bef8e4b45eb9f6ed7734d04da38ced226495548f46e0c8ff8d7d9a524',
   '0x000000000000000000000000af45f76c18d8ae423c5912911dcb8929ec3bbd88'],
  'data': '0x000000000000000000000000000000000000000000000000016345785d8a0000',
  'blockNumber': '0xa2b3f5',
  'blockHash': '0xc1ea3fa0dcf23d2240e78a8

In [3]:
def read_bettor_nonce(rpc_url: str, contract: str, bettor: str) -> int:
    w3 = Web3(Web3.HTTPProvider(rpc_url))
    abi = [
        {
            "inputs": [{"name": "", "type": "address"}],
            "name": "bettorNonces",
            "outputs": [{"name": "", "type": "uint256"}],
            "stateMutability": "view",
            "type": "function",
        }
    ]
    c = w3.eth.contract(address=Web3.to_checksum_address(contract), abi=abi)
    return int(c.functions.bettorNonces(Web3.to_checksum_address(bettor)).call())

nonce = read_bettor_nonce(rpc_url, contract_address, bettor)

In [ ]:
import json
from web3 import Web3
from hexbytes import HexBytes
from web3._utils.events import get_event_data


with open('contract_abi.json', 'r') as f:
    abi = json.load(f)

abi = abi['abi']

w3 = Web3(Web3.HTTPProvider(rpc_url))  # you can also use Web3() just for decode

c = w3.eth.contract(address=Web3.to_checksum_address(contract_address), abi=abi)
bet_abi = [event for event in abi if event.get('name') == 'BetPlaced']

betplaced_topic0 = w3.keccak(
    text="BetPlaced(uint256,address,string,uint8,uint8,int256,uint256,uint256)"
).hex()
for log in logs:
    if not log.get("topics"):
        continue
    topic0 = log["topics"][0].lower().removeprefix("0x")
    bp0 = betplaced_topic0.lower().removeprefix("0x")
    if topic0 == bp0:
        print("BetPlaced found:", log)


HTTPError: 400 Client Error: Bad Request for url: https://eth-sepolia.g.alchemy.com/v2/2hrvMOtTGKhVPYzW59O57

In [14]:
betplaced_topic0

'f9265f6ad9739a6bc5359cbadbef5fd199a55e0679e3a2387f9c80ae28f79603'

In [22]:
def bets_for_bettor(bettor: str):
    decoded_bets = []
    bets = []
    for l in logs:
        if not l.get("topics"):
            continue
        try:
            evt = c.events.BetPlaced().process_log({
                "address": Web3.to_checksum_address(l["address"]),
                "topics": l["topics"],
                "data": l["data"],
                "blockNumber": int(l["blockNumber"], 16),
                "transactionHash": l["transactionHash"],
                "transactionIndex": int(l["transactionIndex"], 16),
                "blockHash": l["blockHash"],
                "logIndex": int(l["logIndex"], 16),
            })
            decoded_bets.append(dict(evt["args"]))
        except Exception:
            # not BetPlaced (or decode mismatch), skip
            pass
    for bet in decoded_bets:
        if bet["bettor"] == bettor:
            bets.append(bet)
    return bets
bets_for_bettor(bettor)

[{'betId': 0,
  'bettor': '0x244e18F228b2804f3a4445d2446c1847b55B294A',
  'gameId': '70506078',
  'betType': 0,
  'outcome': 1,
  'line': 0,
  'amount': 50000000000000000,
  'odds': 216},
 {'betId': 1,
  'bettor': '0x244e18F228b2804f3a4445d2446c1847b55B294A',
  'gameId': '70506090',
  'betType': 0,
  'outcome': 1,
  'line': 0,
  'amount': 1000000000000000,
  'odds': 300}]

In [29]:
c.functions.getGame("70506090").call()

(0, 0, 0, 0, True)